# Run Coding Assistant — Public Network

This notebook demonstrates the coding assistant experience with **all 5 MCP servers** active, leveraging both external services (internet) and internal tools.

**Environment**: Public network (internet access available)

| Tool | Server | Use Case |
|------|--------|----------|
| Context7 | External | Official library documentation (FastAPI, SQLAlchemy, etc.) |
| SearXNG | Self-hosted | Web search — meta-engine, no rate limits |
| Code Sandbox | Local | Execute and verify code snippets |
| Codebase Search | Local AI | Semantic search over internal codebase |
| Repo Docs | Local AI | Q&A over internal architecture/runbook docs |

> **Compare with**: `3_run_closed_coding_assistant.ipynb` — same scenario with only air-gapped tools.

**Scenario**: Add a "daily specials" feature to the `cafe-order-system`.

## 1. Verify MCP Server Connectivity

In [6]:
import subprocess, json

result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

ROUTES = {}
for line in result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        ROUTES[name] = f"https://{host}/mcp"

EXPECTED = ["mcp-context7", "mcp-searxng", "mcp-code-sandbox", "mcp-codebase-search", "mcp-repo-docs"]

print("MCP Servers (Public Network Mode — all 5 active):")
print("=" * 65)
for name in EXPECTED:
    url = ROUTES.get(name, "NOT DEPLOYED")
    status = "READY" if name in ROUTES else "MISSING"
    airgap = "Internet" if name in ("mcp-context7", "mcp-searxng") else "Local"
    print(f"  [{status}] {name:<25} ({airgap})  {url}")

missing = [n for n in EXPECTED if n not in ROUTES]
if missing:
    print(f"\n  WARNING: Missing servers: {missing}")
    print("  Run 1_mcp_servers/2_deploy_mcp_servers.ipynb first.")
else:
    print(f"\n  All 5 servers ready.")

MCP Servers (Public Network Mode — all 5 active):
  [READY] mcp-context7              (Internet)  https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [READY] mcp-searxng                (Internet)  https://mcp-searxng-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [READY] mcp-code-sandbox          (Local)  https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [READY] mcp-codebase-search       (Local)  https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [READY] mcp-repo-docs             (Local)  https://mcp-repo-docs-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp

  All 5 servers ready.


## 2. AGENTS.md — Agent Harness Engineering

[AGENTS.md](https://github.com/agentsmd/agents.md) is an open format that gives AI coding agents **project-specific instructions** — like a README, but for agents. Place it in the project root and every agent (Cursor, Claude Code, OpenCode) reads it automatically.

### Why it matters

Without `AGENTS.md`, the agent relies only on its general knowledge. With it, the agent follows an **inner loop harness** — a plan-execute-verify cycle scoped to *your* project's conventions:

```
┌─────────────────────────────────────────────────────────┐
│             AGENTS.md  Harness  (inner loop)            │
│                                                         │
│   1. Plan        ─→  2. Execute   ─→  3. Verify        │
│   read AGENTS.md     follow            Code Sandbox     │
│   search codebase    conventions       run & test       │
│        ↑                                   │            │
│        └──── 4. Reflect (fix failures) ────┘            │
└─────────────────────────────────────────────────────────┘
```

This is **harness engineering**: the `AGENTS.md` file *is* the harness. It turns a generic coding agent into a project-aware one that verifies its own work.

### Multi-IDE support

| IDE | Reads | Also reads |
|-----|-------|------------|
| Cursor | `AGENTS.md` | `.cursorrules` |
| Claude Code | `AGENTS.md` | `CLAUDE.md` |
| OpenCode | `AGENTS.md` | — |

Since `AGENTS.md` is checked into the repo, all team members' agents follow the same conventions automatically.

In [7]:
from pathlib import Path

agents_md = Path("../0_setup/apps/cafe-order-system/AGENTS.md").read_text()

print("=== cafe-order-system/AGENTS.md ===")
print(agents_md)
print("=" * 50)
print("\nThis file is already included in the cafe-order-system app.")
print("For your own projects, adapt the sections to match your codebase:")
print("  - Project overview & layer architecture")
print("  - Coding conventions (route patterns, DB access, error format)")
print("  - Verification steps (import check, schema test, response format)")
print("\nThe agent reads this file automatically before starting any task.")

=== cafe-order-system/AGENTS.md ===
# AGENTS.md — Cafe Order System

## Project overview

Internal cafe ordering REST API built with **FastAPI + SQLAlchemy + Pydantic**.
Korean-language domain (menu names, descriptions, error messages may be in Korean).

### Layer architecture

```
Router (app/routes/)  →  Service (app/services/)  →  Model (app/models.py)
                                                      Schema (app/schemas.py)
```

| Layer | Location | Responsibility |
|-------|----------|----------------|
| Router | `app/routes/<resource>.py` | HTTP handling, input validation, response formatting |
| Service | `app/services/<name>_service.py` | Business logic (price calculation, inventory checks) |
| Model | `app/models.py` | SQLAlchemy ORM models, enums (`MenuCategory`, `OrderStatus`) |
| Schema | `app/schemas.py` | Pydantic request/response DTOs (`*Create`, `*Response`) |
| Database | `app/database.py` | Session factory, `get_db` dependency |

### Data model

```
MenuItem (1) ─

## 3. Scenario: Add "Daily Specials" to cafe-order-system

We'll walk through how the coding assistant uses **all 5 tools** together to implement a new feature.

### Task
Add a `/api/specials` endpoint that returns today's daily special menu items (discounted items selected by the barista each morning).

---

### Step 1: Understand the existing codebase

**Tool: Codebase Search** — "How are menu items structured?"

In [8]:
# Simulate what the agent does: search internal codebase for menu item structure
import subprocess, json

init = json.dumps({"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}})
call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"search_code","arguments":{"query":"menu item model class definition","top_k":2}}})

url = ROUTES.get("mcp-codebase-search", "")
if url:
    # Initialize
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    # Call tool
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("[Step 1] Agent uses: Codebase Search > search_code")
    print("Query: 'menu item model class definition'")
    print("Purpose: Understand existing code structure before writing new code")
    print("=" * 60)
    # Parse SSE response
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", "")[:800])
            except: pass
else:
    print("mcp-codebase-search not available")

[Step 1] Agent uses: Codebase Search > search_code
Query: 'menu item model class definition'
Purpose: Understand existing code structure before writing new code
--- Result 1 (score: 0.550) ---
File: models.py (lines 71-83)



class OrderItem(Base):
    __tablename__ = "order_items"

    id = Column(Integer, primary_key=True, index=True)
    order_id = Column(Integer, ForeignKey("orders.id"), nullable=False)
    menu_item_id = Column(Integer, ForeignKey("menu_items.id"), nullable=False)
    quantity = Column(Integer, default=1)
    customization = Column(String(200))

    order = relationship("Order", back_populates="items")
    menu_item = relationship("MenuItem", back_populates="order_items")

--- Result 2 (score: 0.503) ---
File: ..2026_06_21_00_31_22.3381324740/schemas.py (lines 1-40)

from datetime import datetime
from typing import Optional

from pydantic import BaseModel, Field

from app.models import MenuCategory, OrderStatus


class MenuIt


### Step 2: Look up FastAPI documentation

**Tool: Context7** — "How to create a FastAPI endpoint with query parameters?"

Context7 fetches the **latest official FastAPI docs** — something only possible with internet access.

In [11]:
url = ROUTES.get("mcp-context7", "")
if url:
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"resolve-library-id","arguments":{"query":"fastapi","libraryName":"fastapi"}}})
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("[Step 2] Agent uses: Context7 > resolve-library-id")
    print("Query: 'fastapi'")
    print("Purpose: Look up official library documentation (internet required)")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", "")[:500])
            except: pass
else:
    print("mcp-context7 not available (requires internet)")

[Step 2] Agent uses: Context7 > resolve-library-id
Query: 'fastapi'
Purpose: Look up official library documentation (internet required)
Available Libraries:

- Title: FastAPI
- Context7-compatible library ID: /fastapi/fastapi
- Description: FastAPI framework, high performance, easy to learn, fast to code, ready for production
- Code Snippets: 2153
- Source Reputation: High
- Benchmark Score: 85.57
- Versions: 0.115.13, 0_116_1, 0.118.2, 0.122.0, 0.128.0
----------
- Title: FastAPI
- Context7-compatible library ID: /websites/fastapi_tiangolo
- Description: FastAPI is a modern, high-performance web framework for building APIs with


### Step 3: Search web for best practices

**Tool: SearXNG** — "FastAPI daily scheduler pattern SQLAlchemy"

SearXNG searches the web for relevant blog posts, StackOverflow answers, and tutorials.

In [ ]:
url = ROUTES.get("mcp-searxng", "")
if url:
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"search-web","arguments":{"query":"FastAPI daily scheduled task SQLAlchemy best practice","max_results":3}}})
    # SearXNG HTTP transport requires session ID from init response
    r_init = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-D","/tmp/sxg_headers.txt","-d",init,"-m","10",url], capture_output=True, text=True)
    session_id = ""
    try:
        with open("/tmp/sxg_headers.txt") as f:
            for line in f:
                if line.lower().startswith("mcp-session-id"):
                    session_id = line.split(":",1)[1].strip()
    except: pass
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-H",f"Mcp-Session-Id: {session_id}","-d",call,"-m","30",url], capture_output=True, text=True)
    print("[Step 3] Agent uses: SearXNG > search-web")
    print("Query: 'FastAPI daily scheduled task SQLAlchemy best practice'")
    print("Purpose: Search web for community best practices and patterns")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", "")[:600])
            except: pass
else:
    print("mcp-searxng not available (requires internet)")

[Step 3] Agent uses: SearXNG > search-web
Query: 'FastAPI daily scheduled task SQLAlchemy best practice'
Error: DDG detected an anomaly in the request, you are likely making requests too quickly.


### Step 4: Check internal documentation

**Tool: Repo Docs** — "What is the API endpoint pattern in this project?"

In [14]:
url = ROUTES.get("mcp-repo-docs", "")
if url:
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"search_docs","arguments":{"query":"API endpoint design pattern and response format","top_k":2}}})
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("[Step 4] Agent uses: Repo Docs > search_docs")
    print("Query: 'API endpoint design pattern and response format'")
    print("Purpose: Check internal docs for project conventions")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", "")[:600])
            except: pass
else:
    print("mcp-repo-docs not available")

[Step 4] Agent uses: Repo Docs > search_docs
Query: 'API endpoint design pattern and response format'
Purpose: Check internal docs for project conventions
--- Result 1 (score: 0.384) ---
Source: ..2026_06_17_08_53_51.704143465/api-guide.md > Health Check

```
GET /health
```

Response:
```json
{"status": "healthy", "service": "Cafe Order System", "version": "1.2.0"}
```

---

--- Result 2 (score: 0.377) ---
Source: api-guide.md > Health Check

```
GET /health
```

Response:
```json
{"status": "healthy", "service": "Cafe Order System", "version": "1.2.0"}
```

---


### Step 5: Execute and verify the implementation

**Tool: Code Sandbox** — Run the generated code to verify it works.

In [15]:
url = ROUTES.get("mcp-code-sandbox", "")
if url:
    test_code = '''import json\nspecials = [{"name": "아메리카노", "price": 3500, "original_price": 4500, "discount": "22%"},\n            {"name": "크루아상", "price": 3000, "original_price": 4000, "discount": "25%"}]\nprint(json.dumps({"date": "2026-06-16", "specials": specials}, ensure_ascii=False, indent=2))'''
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"execute_code","arguments":{"code":test_code,"language":"python"}}})
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("[Step 5] Agent uses: Code Sandbox > execute_code")
    print("Action: Run generated daily specials API response")
    print("Purpose: Verify implementation actually works (inner loop)")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", ""))
            except: pass
else:
    print("mcp-code-sandbox not available")

[Step 5] Agent uses: Code Sandbox > execute_code
Action: Run generated daily specials API response
Purpose: Verify implementation actually works (inner loop)
[python] OK (0.02s)

stdout:
{
  "date": "2026-06-16",
  "specials": [
    {
      "name": "아메리카노",
      "price": 3500,
      "original_price": 4500,
      "discount": "22%"
    },
    {
      "name": "크루아상",
      "price": 3000,
      "original_price": 4000,
      "discount": "25%"
    }
  ]
}


## 4. Harness Test: Agent Autonomously Follows AGENTS.md

The steps above (1-5) were **manual simulations** — we called each MCP tool individually to show what the agent does.

Now let's test the **harness in action**: feed `AGENTS.md` to the model as system context, give it a task, and observe that it **autonomously plans the correct tool usage order** without any step-by-step prompting.

This demonstrates the key value of AGENTS.md: **one file, autonomous agent behavior**.

```
┌────────────────────────────────────────────────┐
│  Input:  AGENTS.md (system) + task (user)      │
│  Output: Agent response following inner loop   │
│          1. Plans which tools to use           │
│          2. Writes code following conventions  │
│          3. Includes verification steps        │
└────────────────────────────────────────────────┘
```

In [ ]:
import os, json, subprocess
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

MODEL_NAME = os.getenv("MODEL_NAME", "qwen36-27b")
CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN", "apps.openshift-cluster.sandbox1785.opentlc.com")
MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", f"https://maas-api.{CLUSTER_DOMAIN}/{MODEL_NAMESPACE}/{MODEL_NAME}")
MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")

agents_md = Path("../0_setup/apps/cafe-order-system/AGENTS.md").read_text()

TASK = "Add a GET /api/specials endpoint that returns today's daily special menu items with discounted prices."

messages = [
    {"role": "system", "content": f"You are a coding agent. Follow the instructions in AGENTS.md exactly.\n\n{agents_md}"},
    {"role": "user", "content": TASK}
]

payload = {
    "model": MODEL_NAME,
    "messages": messages,
    "max_tokens": 2048,
    "temperature": 0.2
}

print("=" * 70)
print("[Harness Test] Sending task to model WITH AGENTS.md as system context")
print("=" * 70)
print(f"  Model:    {MODEL_NAME}")
print(f"  Endpoint: {MODEL_ENDPOINT}/v1")
print(f"  Task:     {TASK}")
print(f"  System:   AGENTS.md ({len(agents_md)} chars)")
print("-" * 70)

r = subprocess.run(
    ["curl", "-sk", "-X", "POST",
     "-H", "Content-Type: application/json",
     "-H", f"Authorization: Bearer {MAAS_API_KEY}",
     "-d", json.dumps(payload),
     "-m", "60",
     f"{MODEL_ENDPOINT}/v1/chat/completions"],
    capture_output=True, text=True
)

if r.returncode == 0:
    try:
        resp = json.loads(r.stdout)
        content = resp["choices"][0]["message"]["content"]
        print("\n[Agent Response — guided by AGENTS.md harness]\n")
        print(content[:3000])
        if len(content) > 3000:
            print(f"\n... (truncated, total {len(content)} chars)")

        print("\n" + "=" * 70)
        print("[Harness Verification] Checking if agent followed the inner loop:")
        checks = {
            "1. Plan (searched codebase/docs first)": any(k in content.lower() for k in ["search", "codebase", "existing", "pattern", "repo docs"]),
            "2. Execute (followed conventions)": any(k in content.lower() for k in ["apirouter", "depends(get_db)", "schemas.py", "httpexception"]),
            "3. Verify (included verification)": any(k in content.lower() for k in ["verify", "test", "import check", "code sandbox", "execute"]),
        }
        for check, passed in checks.items():
            status = "PASS" if passed else "MISS"
            print(f"  [{status}] {check}")
        print("=" * 70)
    except (json.JSONDecodeError, KeyError) as e:
        print(f"Response parse error: {e}")
        print(r.stdout[:500])
else:
    print(f"Request failed (exit code {r.returncode})")
    print(r.stderr[:300] if r.stderr else "No stderr")

In [ ]:
messages_no_harness = [
    {"role": "system", "content": "You are a coding agent."},
    {"role": "user", "content": TASK}
]

payload_no_harness = {
    "model": MODEL_NAME,
    "messages": messages_no_harness,
    "max_tokens": 2048,
    "temperature": 0.2
}

print("=" * 70)
print("[Control Test] Same task WITHOUT AGENTS.md (no harness)")
print("=" * 70)
print(f"  Task:   {TASK}")
print(f"  System: 'You are a coding agent.' (generic, no project context)")
print("-" * 70)

r2 = subprocess.run(
    ["curl", "-sk", "-X", "POST",
     "-H", "Content-Type: application/json",
     "-H", f"Authorization: Bearer {MAAS_API_KEY}",
     "-d", json.dumps(payload_no_harness),
     "-m", "60",
     f"{MODEL_ENDPOINT}/v1/chat/completions"],
    capture_output=True, text=True
)

if r2.returncode == 0:
    try:
        resp2 = json.loads(r2.stdout)
        content2 = resp2["choices"][0]["message"]["content"]
        print("\n[Agent Response — NO harness (generic)]\n")
        print(content2[:2000])
        if len(content2) > 2000:
            print(f"\n... (truncated, total {len(content2)} chars)")

        print("\n" + "=" * 70)
        print("[Comparison] Without AGENTS.md, the agent typically:")
        print("  - Writes generic code (not project-specific patterns)")
        print("  - Skips searching existing codebase first")
        print("  - Does NOT include verification steps")
        print("  - May not follow the project's layer architecture")
        print("=" * 70)
    except (json.JSONDecodeError, KeyError) as e:
        print(f"Response parse error: {e}")
        print(r2.stdout[:500])
else:
    print(f"Request failed (exit code {r2.returncode})")

## 6. Summary: Public Network Tool Usage

| Step | Task | Tool Used | Source |
|------|------|-----------|--------|
| 1 | Understand existing code | **Codebase Search** | Internal codebase (local) |
| 2 | Look up library docs | **Context7** | Official FastAPI docs (internet) |
| 3 | Search best practices | **SearXNG** | Web search — self-hosted meta-engine (internet) |
| 4 | Check internal conventions | **Repo Docs** | Architecture/API docs (local) |
| 5 | Verify implementation | **Code Sandbox** | Local execution (local) |

### Key advantage of Public mode:
- Access to **latest official documentation** via Context7
- Access to **community knowledge** (blogs, SO) via SearXNG — no rate limits
- Combined with internal tools for project-specific context

## Next Steps

- `3_run_closed_coding_assistant.ipynb` — See how the same task works in an air-gapped environment
- `../2_maas/` — Add MaaS gateway for auth and rate limiting on top of these tools